In [ ]:
# ============================================================
# STAGE 1 — U-NET 2D BASELINE: probability maps for halo detection
# ============================================================
# Pipeline (prof's whiteboard):
#   SPH image -> [U-Net] -> CONTINUOUS probability map -> (threshold -> stage 2)
# This notebook trains the U-Net and runs the threshold sweep.
# Stage 2 (connectivity -> catalog) will consume val_prob_patches.npy.

import os
import json
import math
import time
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

DATA_DIR = r"C:\Users\kekko\Downloads\Python\TESI\CAMELS_data"
IN_ROOT  = os.path.join(DATA_DIR, "input_data")      # sim_<id>/X_patches.npy, ...
RUN_ROOT = os.path.join(DATA_DIR, "unet_runs")
RUN_NAME = "unet_v1_baseline"

# Split by SIMULATION (patches of the same sim are correlated -> no leakage).
# sim_ids look like "LH_100_090"; a prefix "LH_100" matches all its snapshots.
SPLIT = {
    "train": ["LH_100", "LH_101", "LH_102"],
    "val":   ["LH_103"],
    "test":  ["LH_104"],        # fill with NEW simulations when downloaded (prof: 5-10 runs)
}

# Model (minimal U-Net, ~2.1M params with BASE_CH=32, DEPTH=3)
BASE_CH = 32
DEPTH   = 3          # pooling levels: 256 -> 32 bottleneck

# Training
BATCH_SIZE   = 8
LR           = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS   = 30
PATIENCE     = 15    # early stopping on validation soft-Dice (threshold-free)
GRAD_CLIP    = 1.0
LOSS_MODE    = "bce" # "bce" (with pos_weight) or "bce+dice"
DICE_WEIGHT  = 0.5
USE_AUGMENTATION = False   # prof: "probably don't need data augmentation"

# Threshold sweep (operating point chosen on the VALIDATION sim)
THRESHOLDS         = np.round(np.arange(0.05, 0.96, 0.05), 2)
MATCH_TOL_PX       = 3.0   # object matching: centroid within max(tol, r_eq_gt)
EXCLUDE_EDGE_BLOBS = True  # ignore blobs cut by the patch border in object metrics

SEED = 42
NUM_WORKERS = 0     # Windows/Jupyter: keep 0

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cpu":
    print("WARNING: training on CPU will be slow. Consider reducing MAX_EPOCHS or using a GPU.")

In [ ]:
# ============================================================
# DATA LOADING AND SPLITTING
# ============================================================

def discover_patch_folders(root):
    if not os.path.isdir(root):
        raise FileNotFoundError(root)
    found = {}
    for name in sorted(os.listdir(root)):
        if not name.startswith("sim_"):
            continue
        d = os.path.join(root, name)
        xp = os.path.join(d, "X_patches.npy")
        yp = os.path.join(d, "Y_mask_patches.npy")
        if os.path.exists(xp) and os.path.exists(yp):
            found[name[4:]] = d
    if not found:
        raise RuntimeError(f"No sim_* folders with X/Y patches in {root}")
    return found

def resolve_split(available, split):
    resolved, used = {}, set()
    for role, keys in split.items():
        ids = []
        for k in keys:
            matches = [s for s in available if s == k or s.startswith(k + "_")]
            if not matches:
                raise KeyError(f"[{role}] '{k}' matches no simulation. Available: {sorted(available)}")
            ids += matches
        dup = used & set(ids)
        if dup:
            raise ValueError(f"Simulations {dup} assigned to multiple roles")
        used |= set(ids)
        resolved[role] = sorted(ids)
    return resolved

def load_split(sim_ids, folders):
    """Concatenate patches of the given sims + per-patch metadata for traceability."""
    Xs, Ys, meta = [], [], []
    for sim_id in sim_ids:
        X = np.load(os.path.join(folders[sim_id], "X_patches.npy"), mmap_mode="r")
        Y = np.load(os.path.join(folders[sim_id], "Y_mask_patches.npy"), mmap_mode="r")
        n_grid = (256 and X.shape[-1] // 256)
        per_slice = n_grid * n_grid
        Xs.append(np.asarray(X, dtype=np.float32))
        Ys.append(np.asarray(Y, dtype=np.float32))
        for i in range(X.shape[0]):
            s, rem = divmod(i, per_slice)
            gy, gx = divmod(rem, n_grid)
            meta.append({"sim_id": sim_id, "patch_idx": int(i),
                         "slice": int(s), "grid_y": int(gy), "grid_x": int(gx)})
    return np.concatenate(Xs), np.concatenate(Ys), meta

FOLDERS = discover_patch_folders(IN_ROOT)
SPLIT_RESOLVED = resolve_split(FOLDERS, SPLIT)

data = {}
for role, ids in SPLIT_RESOLVED.items():
    X, Y, meta = load_split(ids, FOLDERS)
    data[role] = {"X": X, "Y": Y, "meta": meta}
    pos_pct = 100.0 * Y.mean()
    print(f"{role:>5}: {X.shape[0]:>4} patches from {ids} | positive pixels: {pos_pct:.2f}%")

if not SPLIT_RESOLVED["test"]:
    print("\nWARNING: test split is empty. Validation metrics will be reported, but final "
          "thesis numbers need held-out simulations never used for any choice.")

# pos_weight from the TRAINING set only (imbalance ~1:350)
pos = float(data["train"]["Y"].sum())
neg = float(data["train"]["Y"].size - pos)
POS_WEIGHT = neg / max(pos, 1.0)
print(f"\npos_weight (train): {POS_WEIGHT:.1f}")

# Torch tensors
for role in data:
    data[role]["Xt"] = torch.from_numpy(np.ascontiguousarray(data[role]["X"]))
    data[role]["Yt"] = torch.from_numpy(np.ascontiguousarray(data[role]["Y"]))

In [ ]:
# ============================================================
# DATASET AND DATALOADER
# ============================================================

class PatchDataset(Dataset):
    def __init__(self, Xt, Yt, augment=False):
        self.X, self.Y = Xt, Yt
        self.augment = augment

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        x, y = self.X[i], self.Y[i]
        if self.augment:                      # flips/rot90 are physically valid (isotropy)
            k = int(torch.randint(0, 4, (1,)))
            if k:
                x = torch.rot90(x, k, (-2, -1)); y = torch.rot90(y, k, (-2, -1))
            if torch.rand(1) < 0.5:
                x = torch.flip(x, (-1,)); y = torch.flip(y, (-1,))
            if torch.rand(1) < 0.5:
                x = torch.flip(x, (-2,)); y = torch.flip(y, (-2,))
        return x, y

def make_loader(role, shuffle):
    ds = PatchDataset(data[role]["Xt"], data[role]["Yt"], augment=(shuffle and USE_AUGMENTATION))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))

LOADERS = {"train": make_loader("train", True),
           "val":   make_loader("val", False)}

In [ ]:
# ============================================================
# U-NET MODEL
# ============================================================

class DoubleConv(nn.Sequential):
    def __init__(self, c_in, c_out):
        super().__init__(
            nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
            nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1, bias=False),
            nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        )

class Down(nn.Sequential):
    def __init__(self, c_in, c_out):
        super().__init__(nn.MaxPool2d(2), DoubleConv(c_in, c_out))

class Up(nn.Module):
    def __init__(self, c_in):
        super().__init__()
        self.up = nn.ConvTranspose2d(c_in, c_in // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(c_in, c_in // 2)   # after concat with the skip

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_ch=1, base=BASE_CH, depth=DEPTH):
        super().__init__()
        chans = [base * 2 ** i for i in range(depth + 1)]
        self.inc = DoubleConv(in_ch, chans[0])
        self.downs = nn.ModuleList(Down(chans[i], chans[i + 1]) for i in range(depth))
        self.ups = nn.ModuleList(Up(chans[depth - i]) for i in range(depth))
        self.outc = nn.Conv2d(base, 1, kernel_size=1)   # logits; sigmoid applied at inference

    def forward(self, x):
        skips = []
        x = self.inc(x)
        for down in self.downs:
            skips.append(x)
            x = down(x)
        for up, skip in zip(self.ups, reversed(skips)):
            x = up(x, skip)
        return self.outc(x)

model = UNet().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"UNet parameters: {n_params:,}")

In [ ]:
# ============================================================
# LOSS FUNCTION
# ===========================================================


class BCEDiceLoss(nn.Module):
    """BCE(pos_weight) + soft Dice on probabilities."""
    def __init__(self, pos_weight, dice_weight=DICE_WEIGHT, eps=1.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=DEVICE))
        self.dice_weight = dice_weight
        self.eps = eps

    def forward(self, logits, y):
        bce = self.bce(logits, y)
        p = torch.sigmoid(logits)
        inter = (p * y).sum(dim=(1, 2, 3))
        dice = 1.0 - (2 * inter + self.eps) / (p.sum(dim=(1, 2, 3)) + y.sum(dim=(1, 2, 3)) + self.eps)
        return bce + self.dice_weight * dice.mean()

if LOSS_MODE == "bce":
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
elif LOSS_MODE == "bce+dice":
    criterion = BCEDiceLoss(POS_WEIGHT)
else:
    raise ValueError(LOSS_MODE)

print(f"Loss: {LOSS_MODE} (pos_weight={POS_WEIGHT:.1f})")

In [ ]:
# ============================================================
# TRAINING AND EVALUATION
# ============================================================

def train_one_epoch(model, loader, optimizer):
    model.train()
    total, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        if GRAD_CLIP:
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        total += loss.item() * len(xb); n += len(xb)
    return total / n

@torch.no_grad()
def evaluate(model, loader):
    """Global soft-Dice (threshold-free, for early stopping) + pixel metrics @0.5."""
    model.eval()
    total, n = 0.0, 0
    s_p = s_y = s_py = 0.0
    tp = fp = fn = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        total += criterion(logits, yb).item() * len(xb); n += len(xb)
        p = torch.sigmoid(logits)
        s_p += float(p.sum()); s_y += float(yb.sum()); s_py += float((p * yb).sum())
        b = p > 0.5
        ymask = yb.bool()
        tp += float((b & ymask).sum()); fp += float((b & ~ymask).sum()); fn += float((~b & ymask).sum())
    soft = (2 * s_py + 1.0) / (s_p + s_y + 1.0)
    prec = tp / max(tp + fp, 1.0); rec = tp / max(tp + fn, 1.0)
    return {"loss": total / n, "soft_dice": soft,
            "dice50": 2 * tp / max(2 * tp + fp + fn, 1.0),
            "iou50": tp / max(tp + fp + fn, 1.0),
            "prec50": prec, "rec50": rec}

In [ ]:
# ============================================================
# TRAINING LOOP
# ============================================================

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)

history = {"train_loss": [], "val_loss": [], "val_soft_dice": [], "val_iou50": []}
best_dice, bad_epochs, best_epoch = -1.0, 0, -1
t0 = time.time()
RUN_DIR = os.path.join(RUN_ROOT, RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)

for epoch in range(1, MAX_EPOCHS + 1):
    # --- Progress: show which epoch is running
    print(f"[epoch {epoch:3d}/{MAX_EPOCHS}] training...", end="\r", flush=True)
    t_ep = time.time()

    tr = train_one_epoch(model, LOADERS["train"], optimizer)
    va = evaluate(model, LOADERS["val"])
    scheduler.step(va["soft_dice"])

    history["train_loss"].append(tr)
    history["val_loss"].append(va["loss"])
    history["val_soft_dice"].append(va["soft_dice"])
    history["val_iou50"].append(va["iou50"])

    marker = ""
    if va["soft_dice"] > best_dice:
        best_dice, best_epoch, bad_epochs = va["soft_dice"], epoch, 0
        torch.save(
            {"state_dict": model.state_dict(), "epoch": epoch, "val": va,
             "config": {"base": BASE_CH, "depth": DEPTH}},
            os.path.join(RUN_DIR, "best_model.pt"),
        )
        marker = "  <- best"
    else:
        bad_epochs += 1

    print(f"epoch {epoch:3d}/{MAX_EPOCHS} | train {tr:.4f} | val {va['loss']:.4f} | "
          f"softDice {va['soft_dice']:.4f} | IoU@0.5 {va['iou50']:.4f} "
          f"| lr {optimizer.param_groups[0]['lr']:.2e} | {time.time()-t_ep:5.1f}s{marker}")

    if bad_epochs >= PATIENCE:
        print(f"Early stopping at epoch {epoch} (best: {best_epoch}).")
        break

print(f"\nTraining done in {(time.time() - t0)/60:.1f} min | best epoch {best_epoch} "
      f"| val softDice {best_dice:.4f}")

with open(os.path.join(RUN_DIR, "history.json"), "w") as fh:
    json.dump({"history": history, "best_epoch": best_epoch}, fh, indent=2)

In [ ]:
# ============================================================
# PLOT TRAINING HISTORY
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend(); axes[0].set_title("Loss")
axes[1].plot(history["val_soft_dice"], label="val soft-Dice")
axes[1].plot(history["val_iou50"], label="val IoU @0.5")
axes[1].axvline(best_epoch - 1, color="gray", ls="--", lw=1, label=f"best epoch ({best_epoch})")
axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].set_title("Validation metrics")
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# PREDICTION ON VALIDATION SET
# ============================================================

ckpt = torch.load(os.path.join(RUN_DIR, "best_model.pt"), map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"])
model.eval()

@torch.no_grad()
def predict_probs(model, Xt, batch=16):
    out = []
    for i in range(0, len(Xt), batch):
        out.append(torch.sigmoid(model(Xt[i:i + batch].to(DEVICE))).cpu().numpy())
    return np.concatenate(out)

probs_val = predict_probs(model, data["val"]["Xt"])          # (N_val, 256, 256), continuous
np.save(os.path.join(RUN_DIR, "val_prob_patches.npy"), probs_val.astype(np.float32))
with open(os.path.join(RUN_DIR, "val_patch_meta.json"), "w") as fh:
    json.dump(data["val"]["meta"], fh, indent=2)             # traceability: sim_id, slice, grid

print(f"Saved val_prob_patches.npy {probs_val.shape} -> stage 2 will consume this file.")
print("Probability histogram (the continuous map of the prof's sketch):")
plt.figure(figsize=(6, 3.2))
plt.hist(probs_val.ravel(), bins=100, log=True)
plt.xlabel("predicted probability"); plt.ylabel("pixels (log)")
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# THRESHOLD SWEEP (OPERATING POINT SELECTION)
# ============================================================

STRUCT8 = np.ones((3, 3), dtype=bool)
Y_val = data["val"]["Y"].astype(bool)                        # (N,256,256)

def blob_catalogue(lab, n, exclude_edge):
    if n == 0:
        return []
    H, W = lab.shape
    sizes = np.bincount(lab.ravel())
    coms = ndimage.center_of_mass(np.ones_like(lab), lab, range(1, n + 1))
    slices = ndimage.find_objects(lab)
    out = []
    for i in range(n):
        s = slices[i]
        edge = (s[0].start == 0 or s[1].start == 0 or s[0].stop == H or s[1].stop == W)
        if exclude_edge and edge:
            continue
        out.append((coms[i][0], coms[i][1], math.sqrt(sizes[i + 1] / math.pi)))
    return out

def match_blobs(gt, pr, tol_px):
    """Greedy one-to-one matching by centroid distance, closest first.
    Tolerance per GT blob: max(tol_px, equivalent radius)."""
    cand = []
    for i, (cy, cx, r) in enumerate(gt):
        tol = max(tol_px, r)
        for j, (py, px, _) in enumerate(pr):
            d = math.hypot(cy - py, cx - px)
            if d <= tol:
                cand.append((d, i, j))
    cand.sort()
    used_i, used_j, matched = set(), set(), 0
    for d, i, j in cand:
        if i in used_i or j in used_j:
            continue
        used_i.add(i); used_j.add(j); matched += 1
    return matched

def object_level(probs, ys, t):
    n_gt = n_pr = n_m = 0
    for p, y in zip(probs, ys):
        gtl, n1 = ndimage.label(y, structure=STRUCT8)
        prl, n2 = ndimage.label(p > t, structure=STRUCT8)
        gt = blob_catalogue(gtl, n1, EXCLUDE_EDGE_BLOBS)
        pr = blob_catalogue(prl, n2, EXCLUDE_EDGE_BLOBS)
        n_gt += len(gt); n_pr += len(pr)
        n_m += match_blobs(gt, pr, MATCH_TOL_PX)
    comp = n_m / max(n_gt, 1); pur = n_m / max(n_pr, 1)
    return n_gt, n_pr, n_m, comp, pur, 2 * comp * pur / max(comp + pur, 1e-9)

rows = []
print(f"{'t':>5} {'P_pix':>7} {'R_pix':>7} {'IoU':>7} {'F1_pix':>7} | "
      f"{'GT':>5} {'pred':>5} {'match':>6} {'compl':>7} {'purity':>7} {'F1_obj':>7}")
for t in THRESHOLDS:
    b = probs_val > t
    tp = np.logical_and(b, Y_val).sum()
    fp = np.logical_and(b, ~Y_val).sum()
    fn = np.logical_and(~b, Y_val).sum()
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    iou = tp / max(tp + fp + fn, 1); f1p = 2 * prec * rec / max(prec + rec, 1e-9)
    n_gt, n_pr, n_m, comp, pur, f1o = object_level(probs_val, Y_val, float(t))
    rows.append({"t": float(t), "prec_pix": prec, "rec_pix": rec, "iou": iou, "f1_pix": f1p,
                 "n_gt": n_gt, "n_pred": n_pr, "n_match": n_m,
                 "completeness": comp, "purity": pur, "f1_obj": f1o})
    print(f"{t:5.2f} {prec:7.3f} {rec:7.3f} {iou:7.3f} {f1p:7.3f} | "
          f"{n_gt:5d} {n_pr:5d} {n_m:6d} {comp:7.3f} {pur:7.3f} {f1o:7.3f}")

best = max(rows, key=lambda r: r["f1_obj"])
t_star = best["t"]
print(f"\nChosen operating point: t* = {t_star:.2f}  (max object F1 = {best['f1_obj']:.3f}, "
      f"completeness {best['completeness']:.3f}, purity {best['purity']:.3f})")

with open(os.path.join(RUN_DIR, "threshold_sweep.json"), "w") as fh:
    json.dump(rows, fh, indent=2)
with open(os.path.join(RUN_DIR, "chosen_threshold.json"), "w") as fh:
    json.dump({"threshold": t_star, **best}, fh, indent=2)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ts = [r["t"] for r in rows]
axes[0].plot(ts, [r["prec_pix"] for r in rows], label="precision")
axes[0].plot(ts, [r["rec_pix"] for r in rows], label="recall")
axes[0].plot(ts, [r["iou"] for r in rows], label="IoU")
axes[0].set_title("Pixel-level"); axes[0].set_xlabel("threshold"); axes[0].legend()
axes[1].plot(ts, [r["completeness"] for r in rows], label="completeness")
axes[1].plot(ts, [r["purity"] for r in rows], label="purity")
axes[1].plot(ts, [r["f1_obj"] for r in rows], label="F1 obj")
axes[1].axvline(t_star, color="gray", ls="--", lw=1, label=f"t* = {t_star:.2f}")
axes[1].set_title("Object-level"); axes[1].set_xlabel("threshold"); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# VISUALIZATION OF PREDICTIONS AT DIFFERENT POSITIVE FRACTIONS
# # ============================================================

frac = data["val"]["Y"].sum(axis=(1, 2))
pos_idx = np.flatnonzero(frac > 0)
ranked = pos_idx[np.argsort(frac[pos_idx])]
show = [int(ranked[int(q * (len(ranked) - 1))]) for q in [0.25, 0.5, 0.75, 0.95, 1.0]]

fig, axes = plt.subplots(4, len(show), figsize=(3.0 * len(show), 12), squeeze=False)
for col, idx in enumerate(show):
    x = data["val"]["X"][idx, 0]; p = probs_val[idx]; y = Y_val[idx]
    vmin, vmax = np.percentile(x, [0.2, 99.8])
    axes[0, col].imshow(x, cmap="magma", vmin=vmin, vmax=vmax)
    axes[0, col].set_title(f"idx={idx}", fontsize=9)
    axes[1, col].imshow(p, cmap="viridis", vmin=0, vmax=1)
    axes[2, col].imshow(p > t_star, cmap="gray")
    axes[3, col].imshow(x, cmap="magma", vmin=vmin, vmax=vmax)
    if y.any():
        axes[3, col].contour(y.astype(float), levels=[0.5], colors="cyan", linewidths=0.9)
    if (p > t_star).any():
        axes[3, col].contour((p > t_star).astype(float), levels=[0.5], colors="orange", linewidths=0.9)
for r in range(4):
    for c in range(len(show)):
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
axes[0, 0].set_ylabel("X (SPH)"); axes[1, 0].set_ylabel("P(halo)")
axes[2, 0].set_ylabel(f"binary @ t*={t_star:.2f}"); axes[3, 0].set_ylabel("GT (cyan) vs pred (orange)")
plt.tight_layout(); plt.show()